# I. Sanidad Vegetal y Manejo Integrado de Plagas (MIP)

## Fundamento Biológico y Matemático

El modelo central de este módulo es la **acumulación de Grados-Día (GD)**, que describe la
relación entre temperatura y desarrollo de insectos o enfermedades:

$$GD = \sum_{i=1}^{n} \max\left(0, \frac{T_{max,i} + T_{min,i}}{2} - T_{base}\right)$$

El **Índice de Riesgo de Enfermedad (IRE)** combina temperatura y humedad relativa mediante:

$$IRE = \frac{1}{n}\sum_{i=1}^{n} \mathbb{1}\left[T_i \in [T_{opt,min}, T_{opt,max}] \cap HR_i > HR_{umbral}\right] \cdot w_i$$

El **NDVI** (Índice de Vegetación de Diferencia Normalizada) se calcula como:

$$NDVI = \frac{\rho_{NIR} - \rho_{Red}}{\rho_{NIR} + \rho_{Red}} \in [-1, 1]$$

Valores de NDVI < 0.4 en cultivos adultos pueden indicar estrés biótico o abiótico.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.optimize import curve_fit

def generar_serie_meteorologica(dias=365, lat=-33.0, seed=42):
    rng = np.random.default_rng(seed)
    fechas = pd.date_range(start='2023-01-01', periods=dias, freq='D')
    dia_del_anio = np.arange(1, dias + 1)
    temp_base = 16 - abs(lat) * 0.1
    amplitud = 8 + abs(lat) * 0.15
    fase = 2 * np.pi * (dia_del_anio - 15) / 365
    temp_max = temp_base + amplitud * np.cos(fase) + rng.normal(0, 2.5, dias)
    temp_min = temp_base - amplitud * 0.6 * np.cos(fase) + rng.normal(0, 1.8, dias)
    temp_max = np.clip(temp_max, temp_min + 3, 42)
    humedad_rel = 65 - 15 * np.cos(fase + 0.5) + rng.normal(0, 8, dias)
    humedad_rel = np.clip(humedad_rel, 20, 99)
    lluvia_base = 3.5 * (1 + 0.4 * np.sin(fase + np.pi))
    lluvia = rng.exponential(lluvia_base, dias) * rng.binomial(1, 0.35, dias)
    return pd.DataFrame({
        'fecha': fechas,
        'temp_max_c': np.round(temp_max, 1),
        'temp_min_c': np.round(temp_min, 1),
        'temp_media_c': np.round((temp_max + temp_min) / 2, 1),
        'humedad_rel_pct': np.round(humedad_rel, 1),
        'lluvia_mm': np.round(lluvia, 1),
    })

print('Librerías cargadas correctamente.')

## Parámetros del Modelo MIP

In [ ]:
# Parámetros fijos del modelo (valores por defecto del análisis)
temp_base_val = 10.0      # Temperatura Base (°C) — umbral de desarrollo del insecto
umbral_gd_val = 300       # Umbral de GD para alerta de plaga
umbral_hr_val = 75        # Humedad Relativa Umbral para enfermedad (%)
temp_opt_min_val = 15     # Temp óptima mínima para hongo (°C)
temp_opt_max_val = 25     # Temp óptima máxima para hongo (°C)

print(f'Temperatura base: {temp_base_val}°C | Umbral GD: {umbral_gd_val} | '
      f'HR umbral: {umbral_hr_val}% | Temp hongo: {temp_opt_min_val}-{temp_opt_max_val}°C')

In [ ]:
df_clima = generar_serie_meteorologica(dias=365)

# NDVI sintético con curva de crecimiento logística + ruido sensor
t = np.arange(365)
ndvi_base = 0.15 + 0.65 / (1 + np.exp(-0.05 * (t - 120))) * np.exp(-0.003 * (t - 200)**2 / 100)
ndvi_base = np.clip(ndvi_base, 0.1, 0.9)
rng = np.random.default_rng(99)
df_clima['ndvi'] = np.round(ndvi_base + rng.normal(0, 0.03, 365), 3)
df_clima['ndvi'] = np.clip(df_clima['ndvi'], 0.0, 1.0)

df_clima['cond_hongo'] = (
    (df_clima['temp_media_c'] >= temp_opt_min_val) &
    (df_clima['temp_media_c'] <= temp_opt_max_val) &
    (df_clima['humedad_rel_pct'] >= umbral_hr_val)
).astype(int)

df_clima['ire_14d'] = df_clima['cond_hongo'].rolling(14).mean().fillna(0)
df_clima['gd_diario'] = np.maximum(0, df_clima['temp_media_c'] - temp_base_val)
df_clima['gd_acum'] = df_clima['gd_diario'].cumsum()

df_clima.head()

In [ ]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'Temperatura Máx/Mín y Humedad Relativa',
        'NDVI y Grados-Día Acumulados',
        'Índice de Riesgo de Enfermedad (IRE — ventana 14 días)'
    ],
    vertical_spacing=0.08,
    row_heights=[0.35, 0.35, 0.30]
)

fig.add_trace(go.Scatter(
    x=df_clima['fecha'], y=df_clima['temp_max_c'],
    name='T máx (°C)', line=dict(color='#e74c3c', width=1.5),
    fill='tonexty', fillcolor='rgba(231,76,60,0.08)'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df_clima['fecha'], y=df_clima['temp_min_c'],
    name='T mín (°C)', line=dict(color='#3498db', width=1.5)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df_clima['fecha'], y=df_clima['humedad_rel_pct'],
    name='HR (%)', line=dict(color='#27ae60', width=1, dash='dot')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_clima['fecha'], y=df_clima['ndvi'],
    name='NDVI', line=dict(color='#2ecc71', width=2),
    fill='tozeroy', fillcolor='rgba(46,204,113,0.15)'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df_clima['fecha'], y=df_clima['gd_acum'],
    name='GD Acum.', line=dict(color='#e67e22', width=2)
), row=2, col=1)

fig.add_hline(
    y=umbral_gd_val, row=2, col=1,
    line_dash='dash', line_color='red',
    annotation_text=f'Alerta: {umbral_gd_val} GD',
    annotation_position='bottom right'
)

fig.add_trace(go.Bar(
    x=df_clima['fecha'], y=df_clima['ire_14d'],
    name='IRE 14d', marker_color=df_clima['ire_14d'].apply(
        lambda v: '#e74c3c' if v > 0.5 else '#f39c12' if v > 0.25 else '#27ae60'
    )
), row=3, col=1)
fig.add_hline(y=0.5, row=3, col=1, line_dash='dash', line_color='red',
              annotation_text='Riesgo Alto')
fig.add_hline(y=0.25, row=3, col=1, line_dash='dot', line_color='orange',
              annotation_text='Riesgo Moderado')

fig.update_layout(
    height=750, template='plotly_white',
    title='Dashboard MIP — Monitoreo Integrado de Plagas y Enfermedades',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
from IPython.display import display, HTML
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

In [ ]:
dias_alerta_gd = int((df_clima['gd_acum'] >= umbral_gd_val).sum())
fecha_alerta = df_clima.loc[df_clima['gd_acum'] >= umbral_gd_val, 'fecha'].iloc[0] if dias_alerta_gd > 0 else 'No alcanzado'
dias_riesgo_alto = int((df_clima['ire_14d'] > 0.5).sum())
ndvi_min = round(df_clima['ndvi'].min(), 3)

print('=== Resumen del Análisis MIP ===')
print(f'Fecha de alerta por Grados-Día ({umbral_gd_val} GD acum.): {fecha_alerta if isinstance(fecha_alerta, str) else fecha_alerta.strftime("%d/%m/%Y")}')
print(f'Días con Riesgo Alto de Enfermedad (IRE > 0.5): {dias_riesgo_alto} días')
print(f'NDVI mínimo registrado: {ndvi_min}', '⚠️ posible estrés' if ndvi_min < 0.3 else '✓ normal')
print(f'Precipitación anual acumulada: {df_clima["lluvia_mm"].sum():.0f} mm')

## Análisis de Curva de Degradación Cinética de Plaguicidas (Scipy)

Modelo de degradación de primer orden: **C(t) = C₀ · e^(−k·t)**

In [ ]:
dias_deg = np.arange(0, 60, 2, dtype=float)
rng2 = np.random.default_rng(77)
c0_real = 100.0
k_real = 0.045
conc_obs = c0_real * np.exp(-k_real * dias_deg) + rng2.normal(0, 2, len(dias_deg))
conc_obs = np.clip(conc_obs, 0.1, None)

def modelo_degradacion(t, c0, k):
    return c0 * np.exp(-k * t)

popt, pcov = curve_fit(modelo_degradacion, dias_deg, conc_obs, p0=[90, 0.03])
c0_fit, k_fit = popt
dt50 = np.log(2) / k_fit

t_fit = np.linspace(0, 60, 200)
conc_fit = modelo_degradacion(t_fit, *popt)

fig_deg = go.Figure()
fig_deg.add_trace(go.Scatter(
    x=dias_deg, y=conc_obs, mode='markers',
    name='Concentración observada', marker=dict(color='#e74c3c', size=7)
))
fig_deg.add_trace(go.Scatter(
    x=t_fit, y=conc_fit, mode='lines',
    name=f'Ajuste C(t)=C₀·e^(-kt) — DT₅₀={dt50:.1f}d',
    line=dict(color='#3498db', width=2.5)
))
fig_deg.add_hline(y=c0_fit / 2, line_dash='dash', line_color='gray',
                  annotation_text=f'50% de C₀ → DT₅₀ = {dt50:.1f} días')
fig_deg.update_layout(
    title='Curva de Degradación Cinética de Primer Orden (Scipy curve_fit)',
    xaxis_title='Días post-aplicación', yaxis_title='Concentración (ppm)',
    template='plotly_white', height=400
)
print(f'Parámetros estimados: C₀ = {c0_fit:.1f} ppm | k = {k_fit:.4f} d⁻¹ | DT₅₀ = {dt50:.1f} días')
from IPython.display import display, HTML
display(HTML(fig_deg.to_html(include_plotlyjs="cdn", full_html=False)))